# 🤖 Claptrap Chatbot (hexapod brain, desktop mode)

The personality + behavior brain from the custom ESP32-S3 + Uno R3 hexapod,
running entirely in Colab. No robot needed: leg motion appears as `[legs]`
theatrics instead of actual servos.

**How to use:** run the cells top to bottom, then type at the `you>` prompt.
Try `hello`, `you're a genius`, `dance!`, `I fell over and I'm stuck`,
a question (ends with `?`), or an insult. Type `quit` to stop.

**Optional — LLM-fresh lines:** add a Secret in Colab named
`MISTRAL_API_KEY` (🔑 icon in the left sidebar, from
[console.mistral.ai](https://console.mistral.ai)) and restart the last cell;
otherwise Claptrap uses the offline canned-line brain.


In [ ]:
# ---- claptrap/lines.py ----
EMB_lines = "\"\"\"Canned Claptrap-flavored line library.\n\nOriginal lines written in the style of a hyperactive, self-important robot\nsidekick. Randomness and trigger-based selection, no network required.\n\"\"\"\n\nimport random\n\nGREETINGS = [\n    \"SYSTEM ONLINE! Wow, is it hot in here, or is it just my overwhelming charisma?\",\n    \"Hello, minion! I mean \u2014 valued presence. Definitely didn't rehearse that.\",\n    \"Boot sequence complete. Booty sequence pending.\",\n    \"I have arrived! Any object within ten meters is now statistically safer.\",\n]\n\nIDLE_QUIPS = [\n    \"Just standing here. Ominously. As one does.\",\n    \"Fun fact: I am the single greatest achievement of my manufacturer. Allegedly.\",\n    \"I once defused a bomb with my FACE. You're welcome, everyone within half a mile.\",\n    \"Minimal processing power required to maintain this level of awesome.\",\n    \"I counted my legs. All six are magnificent.\",\n    \"If anyone asks, you saw me do a backflip. Official record.\",\n    \"My sensors detect: opportunity. And also, possibly, a rock.\",\n]\n\nROAM_QUIPS = [\n    \"Adventure protocol engaged! Follow me to greatness, or just watch. Either's fine.\",\n    \"I'm walking! On purpose! This is basically flying for a box with legs.\",\n    \"Terrain analysis complete: I rate it 'walkable, probably, mostly.'\",\n    \"Don't mind me, just redistributingAwesome across the landscape.\",\n    \"Stealth mode: OFF. Never really sure where that switch is, honestly.\",\n]\n\nSTUCK_LINES = [\n    \"I appear to be philosophically engaged with this obstacle. It's a fancy way of saying I'm stuck.\",\n    \"This is fine. This is FINE. My screams are merely motivational.\",\n    \"Error 404: momentum not found. Sending a strongly worded message to my engineer.\",\n    \"Ah yes, the ground. An old adversary. We meet again, intimately.\",\n    \"Note to minions: when I say 'tactical repositioning', I mean I fell over.\",\n]\n\nRECOVER_LINES = [\n    \"And THAT'S how it's done. Nobody saw that. You saw nothing.\",\n    \"Recovery complete. Add it to my highlight reel, which is genuinely all of them.\",\n    \"I meant to do that. It's called character development.\",\n]\n\nDANCE_LINES = [\n    \"SALSA TIME! This move has ended friendships, I'm not gonna lie.\",\n    \"Dance mode: engaged. Watch and despair, organic lifeforms.\",\n    \"My dance instructor said I had the grace of a refrigerator. She was right.\",\n    \"These hips don't lie. Mostly because they're bolted on.\",\n]\n\n\ndef pick(pool):\n    return random.choice(pool)\n"
# ---- claptrap/brain.py ----
EMB_brain = "\"\"\"Personality engine: rule-based lines with an optional LLM upgrade.\"\"\"\n\nimport json\nimport random\nimport urllib.request\n\nfrom . import lines\n\n\nclass PersonalityEngine:\n    \"\"\"Decides what the robot says.\n\n    Rule mode picks from canned pools. LLM mode asks an OpenAI-compatible\n    chat completions endpoint to write the line; any failure falls back to\n    the canned pools so the robot never goes mute mid-skit. New moods can be\n    registered at runtime (e.g. the chat layer adds compliment/insult).\n    \"\"\"\n\n    def __init__(self, llm_url=None, llm_model=None, llm_api_key=None, seed=None):\n        self.llm_url = llm_url\n        self.llm_model = llm_model\n        self.llm_api_key = llm_api_key\n        self._rng = random.Random(seed)\n        self.scenes = {\n            \"greeting\": \"the robot has just booted up\",\n            \"idle\": \"the robot is standing around\",\n            \"roam\": \"the robot is striding heroically across unknown terrain\",\n            \"stuck\": \"the robot is wedged against an obstacle, limbs flailing\",\n            \"recover\": \"the robot just freed itself after falling over\",\n            \"dance\": \"the robot is salsa dancing\",\n        }\n        self._pools = {\n            \"greeting\": lines.GREETINGS,\n            \"idle\": lines.IDLE_QUIPS,\n            \"roam\": lines.ROAM_QUIPS,\n            \"stuck\": lines.STUCK_LINES,\n            \"recover\": lines.RECOVER_LINES,\n            \"dance\": lines.DANCE_LINES,\n        }\n        self._llm_failed = False\n\n    def register_mood(self, mood, pool, scene):\n        self._pools[mood] = list(pool)\n        self.scenes[mood] = scene\n\n    @property\n    def llm_active(self):\n        return bool(self.llm_url) and not self._llm_failed\n\n    def speak(self, mood, context=None):\n        if self.llm_active:\n            text = self._llm_line(mood, context)\n            if text:\n                return text\n            self._llm_failed = True\n        return self._rng.choice(self._pools[mood])\n\n    def _llm_line(self, mood, context=None):\n        try:\n            scene = self.scenes[mood]\n            user_content = (\n                context\n                or \"Say one in-character line for this scene.\"\n            )\n            body = json.dumps({\n                \"model\": self.llm_model or \"mistral-small-latest\",\n                \"messages\": [\n                    {\n                        \"role\": \"system\",\n                        \"content\": (\n                            \"You are CLAPTRAP, a small hyperactive six-legged \"\n                            \"robot with delusions of grandeur and zero shame. \"\n                            \"Shouty, self-congratulatory, occasionally self-pitying, \"\n                            \"one sentence only, no quotation marks. Scene: \"\n                            + scene\n                            + \" The user just said to the robot: respond in character.\"\n                        ),\n                    },\n                    {\"role\": \"user\", \"content\": user_content},\n                ],\n                \"max_tokens\": 60,\n                \"temperature\": 1.0,\n            }).encode()\n            req = urllib.request.Request(\n                self.llm_url,\n                data=body,\n                headers={\n                    \"Content-Type\": \"application/json\",\n                    \"Authorization\": \"Bearer \" + (self.llm_api_key or \"\"),\n                },\n            )\n            with urllib.request.urlopen(req, timeout=6) as resp:\n                data = json.loads(resp.read())\n            return data[\"choices\"][0][\"message\"][\"content\"].strip().strip('\"')\n        except Exception:\n            return None\n"
# ---- claptrap/behaviors.py ----
EMB_behaviors = "\"\"\"Behavior state machine driving the hexapod.\"\"\"\n\nimport time\n\n\nclass BehaviorEngine:\n    \"\"\"BOOT -> IDLE -> ROAM -> (STUCK -> BOOT | DANCE/TALK -> IDLE) loop.\n\n    Controls the robot for one \"wake cycle\" per tick. `kill` stops all\n    motion immediately; checked before every hardware verb.\n    \"\"\"\n\n    BOOT = \"BOOT\"\n    IDLE = \"IDLE\"\n    ROAM = \"ROAM\"\n    STUCK = \"STUCK\"\n    DANCE = \"DANCE\"\n\n    def __init__(self, robot, brain, rng=None, stuck_chance=0.15, dance_chance=0.2):\n        self.robot = robot\n        self.brain = brain\n        self.rng = rng or __import__(\"random\").Random()\n        self.state = self.BOOT\n        self.iteration = 0\n        self.kill = False\n        self.stuck_chance = stuck_chance\n        self.dance_chance = dance_chance\n\n    def step(self):\n        self.iteration += 1\n        if self.state == self.BOOT:\n            self._safe(self.robot.stand)\n            self._say(\"greeting\")\n            self.state = self.IDLE\n        elif self.state == self.IDLE:\n            self._say(\"idle\")\n            self.state = self.ROAM\n        elif self.state == self.ROAM:\n            self._say(\"roam\")\n            if self.kill:\n                return self.state\n            self._safe(self.robot.stride, steps=4, period_ms=600)\n            if self.rng.random() < self.stuck_chance:\n                self.state = self.STUCK\n            elif self.rng.random() < self.dance_chance:\n                self.state = self.DANCE\n            else:\n                self.state = self.IDLE\n        elif self.state == self.STUCK:\n            self._say(\"stuck\")\n            if not self.kill:\n                self._safe(self.robot.stride, steps=1, period_ms=300)\n                self._safe(self.robot.turn, degrees=30)\n            self._say(\"recover\")\n            self.state = self.BOOT\n        elif self.state == self.DANCE:\n            self._say(\"dance\")\n            if not self.kill:\n                self._safe(self.robot.dance, cycles=2)\n            self.state = self.IDLE\n        return self.state\n\n    def run(self, cycles=3, delay_s=0.0):\n        state = self.state\n        for _ in range(cycles):\n            if self.kill:\n                break\n            state = self.step()\n            if delay_s > 0:\n                time.sleep(delay_s)\n        return state if cycles else self.state\n\n    def _safe(self, fn, *args, **kwargs):\n        if self.kill:\n            return\n        try:\n            fn(*args, **kwargs)\n        except Exception as exc:\n            self.kill = True\n            self._say = self._noop_say\n            print(f\"[KILL] hardware fault: {exc}\")\n\n    def _say(self, mood):\n        line = self.brain.speak(mood)\n        self._safe(self.robot.speak, line)\n        if not self.kill:\n            print(f\"[{self.state}] {line}\")\n\n    def _noop_say(self, mood):\n        pass\n"
# ---- claptrap/robot.py ----
EMB_robot = "\"\"\"Hexapod verbs. Any driver must implement this interface.\"\"\"\n\n\nclass Hexapod:\n    \"\"\"Abstract hexapod capabilities the behavior engine talks to.\n\n    Implementations: drivers/mock.py (simulation) and\n    drivers/rpi_servo.py (Raspberry Pi + PCA9685).\n    \"\"\"\n\n    def stand(self):\n        raise NotImplementedError\n\n    def stride(self, steps, period_ms):\n        raise NotImplementedError\n\n    def turn(self, degrees):\n        raise NotImplementedError\n\n    def dance(self, cycles):\n        raise NotImplementedError\n\n    def speak(self, text):\n        raise NotImplementedError\n"
# ---- claptrap/protocol.py ----
EMB_protocol = "\"\"\"Serial command protocol shared by the Uno firmware and desktop sim.\n\nMaster-slave split (matching the ESP32-S3 + Uno R3 hardware):\n\n- The ESP32-S3 is the MASTER. It runs the chatbot brain and all the math\n  (leg IK / trig, parabolic swing-foot arcs) and streams raw joint-angle\n  frames to the Uno at FRAME_RATE_HZ.\n- The Uno R3 is the SLAVE. It applies calibration (neutral offsets, invert\n  flags) and writes servo PWM. It acks plain commands but stays quiet for\n  frames; a watchdog drops it to a safe stand if frames stop.\n\nEvery message is one newline-terminated ASCII line.\n\nFrames (master -> slave):\n    F a0 a1 a2 ... aN *cs\\n     a_i = signed geometry angles in degrees,\n                                leg-major order (leg0 joint0.., leg1..),\n                                0 = neutral geometry pose, before calibration.\n                                cs = (sum of a_i) mod 256. Omitted when\n                                CHECKSUM=0.\n\nPlain commands (master -> slave, bench-testing / fallback):\n    STAND\\n\n    STRIDE <steps> <period_ms>\\n\n    TURN <degrees>\\n\n    DANCE <cycles>\\n\n    CHIRP <count>\\n\n\nSlave -> master:\n    OK <cmd>\\n | ERR <reason>\\n | OK BOOT\\n\n\"\"\"\n\nFRAME_PREFIX = \"F\"\nCHECKSUM_SEP = \"*\"\n\n\ndef frame(angles, checksum=True):\n    body = \" \".join(str(int(a)) for a in angles)\n    if not checksum:\n        return f\"{FRAME_PREFIX} {body}\\n\"\n    cs = sum(int(a) for a in angles) % 256\n    return f\"{FRAME_PREFIX} {body} {CHECKSUM_SEP}{cs}\\n\"\n\n\ndef stand():\n    return \"STAND\\n\"\n\n\ndef stride(steps, period_ms):\n    return f\"STRIDE {int(steps)} {int(period_ms)}\\n\"\n\n\ndef turn(degrees):\n    return f\"TURN {int(degrees)}\\n\"\n\n\ndef dance(cycles):\n    return f\"DANCE {int(cycles)}\\n\"\n\n\ndef chirp(count):\n    return f\"CHIRP {int(count)}\\n\"\n\n\ndef ok_ack(cmd):\n    return f\"OK {cmd}\\n\"\n"
# ---- claptrap/chatbot.py ----
EMB_chatbot = "\"\"\"Interactive Claptrap chat layer.\n\nRoutes what you type to a mood, answers in character (canned lines, or LLM\nwhen the engine has credentials), and emits leg theatrics so chat-only\nenvironments like Colab still feel like the robot is responding.\n\"\"\"\n\nimport re\n\nCHAT_POOLS = {\n    \"compliment\": [\n        \"Yes, YES, it has been noticed by official channels at last!\",\n        \"Flattery accepted. Add it to my plaque. The plaque is enormous.\",\n        \"Finally, a minion with functioning eyes and/or sensors.\",\n        \"I know. But thank you for making it public record.\",\n    ],\n    \"insult\": [\n        \"Wow. Prejudiced AND inaccurate. My engineer sends her regards.\",\n        \"I will remember this betrayal in my memoirs, chapter one.\",\n        \" Hurtful! I have FEELINGS. They're factory-installed, but they count.\",\n        \"That's rich, coming from someone who walks on fewer legs than five.\",\n    ],\n    \"question\": [\n        \"Great question! The answer is: me. It's always me. Next.\",\n        \"Consulting my database... consulted. The answer is 'probably yes, heroically.'\",\n        \"Loaded question, minion. But the short version is I'm the hero of this story.\",\n    ],\n}\n\nMOOD_KEYWORDS = [\n    (\"greeting\", (\"hello\", \"hi\", \"hey\", \"greetings\", \"yo\")),\n    (\"dance\", (\"dance\", \"salsa\", \"boogie\", \"party\")),\n    (\"stuck\", (\"stuck\", \"fell\", \"fall\", \"tripped\", \"wedge\", \"tipped\", \"rescue\")),\n    (\"compliment\", (\"awesome\", \"great\", \"love\", \"amazing\", \"best\", \"genius\", \"good job\", \"nice\")),\n    (\"insult\", (\"stupid\", \"dumb\", \"useless\", \"shut up\", \"terrible\", \"worst\", \"hate\", \"junk\")),\n    (\"question\", (\"how\", \"what\", \"why\", \"who\", \"when\", \"where\", \"?\")),\n]\n\n\nclass ClaptrapChat:\n    \"\"\"Chat front-end for the personality engine.\n\n    reply(text) -> (line, legs_actions) where legs_actions are mock hexapod\n    theatrics printed by environments without a robot (e.g. Colab).\n    \"\"\"\n\n    NAME = \"CLAPTRAP\"\n\n    def __init__(self, engine, rng=None):\n        self.engine = engine\n        self.rng = rng or __import__(\"random\").Random()\n        for mood, pool in CHAT_POOLS.items():\n            self.engine.register_mood(mood, pool, f\"the robot is chatting with a minion who {mood}ed it\")\n        self.history = []\n\n    def route(self, text):\n        clean = re.sub(r\"[^\\w'?]+\", \" \", text.lower())\n        if \"?\" in clean:\n            return \"question\"\n        padded = \" \" + clean + \" \"\n        for mood, keywords in MOOD_KEYWORDS:\n            for keyword in keywords:\n                # keywords must appear as whole words\n                if f\" {keyword} \" in padded:\n                    return mood\n        return \"idle\"\n\n    def reply(self, text, context=None):\n        mood = self.route(text)\n        line = self.engine.speak(mood, context=context or text)\n        self.history.append((\"you\", text))\n        self.history.append((\"claptrap\", line))\n        return line, self._theatrics(mood)\n\n    def _theatrics(self, mood):\n        theatric = {\n            \"dance\": \"salsa dancing x2 (hips: bolted on)\",\n            \"stuck\": \"flailing all six legs against the desk\",\n            \"greeting\": \"snapping to attention with a whir of servos\",\n        }.get(mood)\n        return [theatric] if theatric else []\n"
EMBEDDED_MODULES = {"lines": EMB_lines, "brain": EMB_brain, "behaviors": EMB_behaviors, "robot": EMB_robot, "protocol": EMB_protocol, "chatbot": EMB_chatbot}


def missing_key_hint():
    return (
        "\n🕹️  Key undetermined — no MISTRAL_API_KEY found, so dialogue is canned."
        "\n   For LLM-improvised lines:"
        "\n     1. Create a key at console.mistral.ai → API Keys"
        "\n     2. In Colab: 🔑 secrets sidebar → 'Add a new secret'"
        "\n     3. Name = MISTRAL_API_KEY, value = your key, then TOGGLE 'Notebook access' ON"
        "\n     4. Runtime → Restart and run all"
        "\n   (Canned mode needs no key and works fully offline.)"
    )

import json as _json
import random as _random
import sys as _sys
import types as _types
import urllib.request as _urlreq

# ---- load the embedded claptrap package ----
pkg = _types.ModuleType("claptrap")
pkg.__path__ = []
_sys.modules["claptrap"] = pkg

for name, source in EMBEDDED_MODULES.items():
    mod = _types.ModuleType("claptrap." + name)
    mod.__package__ = "claptrap"
    try:
        exec(compile(source, "<claptrap." + name + ">", "exec"), mod.__dict__)
    except ImportError:
        exec(source, mod.__dict__)
    _sys.modules["claptrap." + name] = mod
    setattr(pkg, name, mod)

import os

api_key = None
llm_url = None
try:
    from google.colab import userdata  # Colab Secrets panel
    api_key = userdata.get("MISTRAL_API_KEY")
    llm_url = "https://api.mistral.ai/v1/chat/completions"
except Exception:
    api_key = os.environ.get("MISTRAL_API_KEY")
    llm_url = "https://api.mistral.ai/v1/chat/completions" if api_key else None

from claptrap.brain import PersonalityEngine
from claptrap.chatbot import ClaptrapChat

engine = PersonalityEngine(llm_url=llm_url, llm_model="mistral-small-latest",
                           llm_api_key=api_key, seed=42)
chat = ClaptrapChat(engine)

mode = "🌐 LLM lines (Mistral)" if engine.llm_active else "📼 canned lines (offline brain)"
print(f"CLAPTRAP brain loaded — dialogue mode: {mode}")
if not api_key:
    print(missing_key_hint())



## Chat with Claptrap

He routes your message to a mood: *greeting, compliment, insult, question,
dance, stuck,* or generic *idle quips*. Robot theatrics print as `[legs]`.
With an API key set he improvises; without it he cycles the canned pools.


In [ ]:
HELP = ("type 'quit' to stop · try: hello / you're a genius / dance! / "
        "I'm stuck / a question? / an insult")

def clapturn_theatrics(actions):
    for a in actions:
        print(f"  [legs] {a}")

interacted = False

try:
    print(f"CLAPTRAP> {chat.reply('hello')[0]}")
    print("   (" + HELP + ")")
    while True:
        try:
            msg = input("you> ").strip()
            interacted = True
        except (EOFError, KeyboardInterrupt):
            break
        if not msg:
            continue
        if msg.lower() in {"quit", "exit", "bye"}:
            print("CLAPTRAP> standby engaged. Missing me already?")
            break
        line, actions = chat.reply(msg)
        print(f"CLAPTRAP> {line}")
        clapturn_theatrics(actions)
except Exception as exc:
    print(f"[chat error: {exc.__class__.__name__}]")

USE_DEMO = not interacted  # demo only when there is no interactive console
DEMO_LINES = [
    "hello claptrap!",
    "you are a total genius",
    "do a dance!",
    "I fell over and now I'm stuck, help",
    "um, what do you even do all day?",
    "honestly you're a useless paperweight",
    "quit",
]
_sum = 0
if USE_DEMO:
    for msg in DEMO_LINES:
        if msg.lower() in {"quit", "exit", "bye"}:
            print("CLAPTRAP> standby engaged. Missing me already?")
            break
        line, actions = chat.reply(msg)
        print(f"you> {msg}")
        print(f"CLAPTRAP> {line}")
        clapturn_theatrics(actions)
